# Sparse labelling of waterhole surface state

Paints class labels onto the **GeoTIFF's own grid**. The display panels are rendered live
from the raster and the mask is written back at exactly that size and transform — nothing
is resampled, because a resampled label is a wrong label.

**Sparse by design.** You are not filling in whole tiles. Paint a few confident patches per
class and move on. At 10 m most basin margins are mixed pixels, and leaving them as class 0
is the correct answer, not an unfinished job.

**Month stepping keeps your work.** Left/right arrows walk through the same site's months so
you can see what a pixel did before and after, without losing any label buffer. Nothing
touches disk until you press `ctrl+s`.

The tool opens in a **separate window**, not inline — painting needs a responsive canvas.

## Setup

`%matplotlib qt` opens a native Qt window. If Qt misbehaves on your machine, `%matplotlib
macosx` also works. Inline backends will not work: they cannot deliver mouse-drag events.

In [ ]:
%matplotlib qt

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wh_config
import wh_footprint
import wh_inventory
import wh_label

cfg = wh_config.load()
manifest = wh_inventory.load_manifest(cfg)

print("config", cfg.source_path.name, "hash", cfg.hash)
print(f"{len(manifest):,} chips, {manifest['site_id'].nunique()} sites")
print("\nclass scheme:")
for definition in cfg.classes:
    print(f"  [{definition.key}] {definition.id} {definition.name:24s} {definition.colour}")

## Parameters

Display and brush settings, here rather than in the YAML so they are visible while you work.

The **class scheme itself stays in `waterhole_seg_config.yaml`** — training and prediction
have to agree with it, and every saved mask records its `scheme_version`. Change classes
there, not here.

In [ ]:
PARAMS = wh_label.LabelParams(
    # Panels, left to right. All share pan and zoom. The last is the free slot —
    # NDMI is there because it responds to water UNDER a canopy, which is the case
    # MNDWI gets wrong when sedges cover a waterhole.
    panels=("rgb", "mndwi", "ndvi", "ndti", "ndmi"),

    # True-colour stretch, fixed so a site looks the same month to month.
    rgb_bands=("B4", "B3", "B2"),
    rgb_max_reflectance=0.30,
    rgb_gamma=0.85,

    # Tight display ranges. Contrast lands on the scene rather than being spent on
    # [-1, 1] values that never occur.
    display_ranges={
        "mndwi": (-0.8, 0.6),
        "ndwi": (-0.8, 0.6),
        "ndvi": (-0.1, 0.9),
        "ndti": (-0.4, 0.4),
        "ndmi": (-0.6, 0.6),
    },

    brush_radius_px=2,
    max_brush_radius_px=20,
    undo_depth=200,
    label_alpha=0.55,

    # Basins are a small part of a 1.5 km chip; opening fully zoomed out wastes time.
    initial_zoom_half_width_px=40,

    labeller_name="scott",
)

print(wh_label.KEY_HELP)

## Build the work queue

Filters the manifest down to tiles worth labelling. Poorly observed chips are excluded by
default — labelling a median built from one cloudy scene teaches the classifier noise.

`months_per_site` thins each site to that many evenly spaced months across its record. This
matters for two reasons. With grouped-by-site cross-validation the effective sample size is
the number of labelled **sites**, not pixels, so breadth across sites beats depth within
one. And queueing every month of a site would make `n` (next tile) identical to `right`
(next month), which is confusing to navigate.

Set `months_per_site=None` to queue every qualifying month.

In [ ]:
SITES = ["025", "075", "000"]      # None for all sites
MONTHS = None                       # e.g. ["2024-03", "2024-09"], or None for all

queue = wh_label.build_queue(
    manifest,
    sites=SITES,
    months=MONTHS,
    max_gap_fraction=0.20,   # skip chips more than 20% unobserved
    min_mean_obs=2.0,        # skip medians resting on fewer than 2 scenes
    months_per_site=6,       # evenly spread across years and seasons; None = every month
)

print(f"{len(queue)} tiles queued across {queue['site_id'].nunique()} sites")
queue.groupby("site_id")["year_month"].apply(lambda s: ", ".join(s))

### Optional: load basin footprints

If you have run `footprint_estimation.ipynb`, the footprint outline is drawn on every panel
(toggle with `f`). It is a guide for where the basin is, not a constraint on where you can
paint — label what you see, not what the footprint says.

In [ ]:
footprints = {}
for site_id in queue["site_id"].unique():
    try:
        footprints[site_id] = wh_footprint.load_mask(cfg, site_id)
    except FileNotFoundError:
        pass

print(f"loaded {len(footprints)} footprint(s) of {queue['site_id'].nunique()} queued sites")

## Launch

Opens the labelling window. Keep this notebook running while you work — closing the kernel
discards any unsaved buffers.

### How to label well

1. **Step through the months first** (left/right) before painting anything. Ambiguous
   Octobers become obvious once you have seen the same pixels in February.
2. **Watch the readout** in the bottom right — band values, every index, and `n_obs` for
   the pixel under the cursor. If `n_obs` is 1, be sceptical of what you are seeing.
3. **Paint conservatively.** A few tens of confident pixels per class per tile is plenty.
   Grouped-by-site cross-validation means your effective sample size is the number of
   labelled *sites*, not pixels — so breadth across sites beats depth within one.
4. **Spend your effort on the hard classes.** Open water and surrounding vegetation will be
   generated automatically by the pseudo-labeller. Aquatic vegetation, wet mud and pugged
   margin are what actually need your judgement.
5. **Save before moving to the next tile** (`ctrl+s`). The status bar marks unsaved tiles
   with `*`.

In [ ]:
labeller = wh_label.launch(queue, manifest, cfg, PARAMS, footprints=footprints)

## Session progress

Run this after labelling to see what has been written to disk.

In [ ]:
label_dir = cfg.paths["labels"]
sidecars = sorted(label_dir.glob("*_labels.json")) if label_dir.exists() else []

import json as _json
records = []
for path in sidecars:
    meta = _json.loads(path.read_text())
    row = {
        "site_id": meta["site_id"],
        "year_month": meta["year_month"],
        "labeller": meta["labeller"],
        "source": meta["source"],
        "n_labelled": meta["n_labelled"],
    }
    row.update(meta["pixel_counts"])
    records.append(row)

if records:
    progress = pd.DataFrame(records).sort_values(["site_id", "year_month"])
    print(f"{len(progress)} labelled tiles across {progress['site_id'].nunique()} sites")
    print(f"{progress['n_labelled'].sum():,} labelled pixels total\n")
    class_names = [d.name for d in cfg.classes if not d.ignore]
    print("pixels per class:")
    print(progress[class_names].sum().sort_values(ascending=False).to_string())
    display(progress)
else:
    print("no labels saved yet")

### Class balance

Surrounding vegetation will vastly outnumber everything else, which is expected and handled
by class weighting at training time. What matters here is whether the *hard* classes —
aquatic vegetation, mud, dry bare — have enough labelled **sites** behind them, not enough
pixels. A class present at only one site cannot be cross-validated.

In [ ]:
if records:
    class_names = [d.name for d in cfg.classes if not d.ignore]
    per_class = pd.DataFrame({
        "pixels": progress[class_names].sum(),
        "tiles": (progress[class_names] > 0).sum(),
        "sites": [
            progress.loc[progress[name] > 0, "site_id"].nunique()
            for name in class_names
        ],
    }).sort_values("sites")
    display(per_class)

    thin = per_class[per_class["sites"] < 3]
    if not thin.empty:
        print("\nclasses labelled at fewer than 3 sites — not yet cross-validatable:")
        print(", ".join(thin.index))
else:
    print("no labels saved yet")

---

### If chips fail to open

The tiles sit on a OneDrive share with Files On-Demand, which dehydrates files to
placeholders. A dehydrated chip reports a normal size but cannot be opened, surfacing as
`errno 60` or *"not recognized as a supported file format"*. Retrying does not help.

Fix: right-click `cookie-cutting` in Finder → **Always Keep on This Device**, or move
`images_tif_v2` off OneDrive. `wh_tiles.read_tile` detects this and says so explicitly.